# Early Dementia Detection from Brain MRI
### DenseNet121 — Staged Transfer Learning Benchmark
**Author:** Claude (Adaptive AI Collaborator)
**Methodology:**
* **Data:** OASIS MRI Dataset, same patient-level splits as the CNN and YOLOv8 benchmarks (`src/data_pipeline.py`).
* **Architecture:** DenseNet121 pretrained on ImageNet, fine-tuned in two stages — a frozen-backbone head-training stage, then a partial-unfreeze fine-tuning stage at a much lower learning rate.
* **Robustness:** Same patient-level split and hybrid resampling as the other two models, so all three are directly comparable on identical test patients.
* **Goal:** Add a third model to the diagnostic benchmark alongside the Custom 6-Block CNN (75.73%/0.4257/0.6680) and YOLOv8 (91.64%/0.7869/0.8447).


## 1. Setup


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "app.py").is_file():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root (looked for src/ and app.py) starting from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.append(str(REPO_ROOT / "src"))

from data_pipeline import CATEGORIES, create_metadata_df, patient_level_split, hybrid_resample
from evaluate import evaluate_predictions

CLASS_NAMES = list(CATEGORIES.keys())

print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))


## 2. Load Metadata & Patient-Level Splits
Same `src/data_pipeline.py` used by the CNN and YOLOv8 notebooks — stratified per-category patient split (with `Moderate Dementia`'s 2 patients reused across all three splits) so this model is benchmarked on the exact same patients as the other two.


In [ ]:
# Update this to wherever the OASIS `Data/<category>/...jpg` folders live on disk
DATA_PATH = REPO_ROOT / "dataset" / "Data"

df = create_metadata_df(DATA_PATH)
train_df, val_df, test_df = patient_level_split(df)

print(f"Total scans: {len(df)} | Unique patients: {df['patient_id'].nunique()}")
for name, split_df in [("Training", train_df), ("Validation", val_df), ("Testing", test_df)]:
    print(f"{name:11s}: {split_df['patient_id'].nunique()} patients | {len(split_df)} scans | classes: {sorted(split_df['category'].unique())}")


## 3. Hybrid Resampling (Training Split Only)
Same 137:1 imbalance fix as the other two models: under-sample majority classes, over-sample minority classes to 8000 each, applied only to training data.


In [ ]:
train_df_balanced = hybrid_resample(train_df, target_samples_per_class=8000)

print("Balanced training distribution:")
print(train_df_balanced["category"].value_counts())


## 4. Data Generators — DenseNet-Specific Preprocessing
DenseNet121 expects ImageNet-style normalization (scaled to `[0, 1]`, then normalized by ImageNet's per-channel mean/std), not the simple `rescale=1./255` used for the from-scratch CNN. We pass `preprocess_input` as the generator's `preprocessing_function` instead of setting `rescale`, so images arrive in exactly the distribution the pretrained backbone was trained on.

We keep `IMG_SIZE=(128, 128)` — smaller than DenseNet's typical 224×224 — for consistency with the CNN and YOLOv8 baselines (same input resolution across all three models keeps the comparison fair), and because MRI slices don't carry the fine ImageNet-style texture detail that would benefit from higher resolution anyway.


In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df_balanced,
    x_col="path",
    y_col="category",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=True,
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="path",
    y_col="category",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="path",
    y_col="category",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False,
)


## 5. Build DenseNet121 with a Custom Classification Head
We load DenseNet121 pretrained on ImageNet with `include_top=False` (dropping its original 1000-class ImageNet head), freeze it initially, and attach a small classification head: `GlobalAveragePooling2D -> Dense(256, relu) -> Dropout(0.5) -> Dense(4, softmax)`.

Note `base_model(inputs, training=False)`: this keeps DenseNet's BatchNormalization layers running in inference mode (using their pretrained moving statistics) throughout — including later in Stage 2 when we set `base_model.trainable = True`. This is the standard recommendation from Keras' transfer-learning guide: recomputing BatchNorm statistics from a small, resampled medical-imaging batch would otherwise destroy the pretrained normalization the backbone relies on.


In [ ]:
def build_densenet_model(input_shape=(128, 128, 3), num_classes=4):
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=input_shape)
    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs), base_model

model, base_model = build_densenet_model()
model.compile(optimizer=Adam(learning_rate=1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


## 6. Stage 1 — Train the Classification Head (Frozen Backbone)
With DenseNet121 frozen, only the new head (~260K params) trains. This lets the head adapt to MRI features before we risk disturbing the pretrained backbone's weights with fine-tuning. Same early-stopping philosophy as the CNN: `EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True)`.


In [ ]:
stage1_checkpoint = ModelCheckpoint(
    "densenet_stage1_head.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    verbose=1,
)
stage1_early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)

history_stage1 = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=[stage1_checkpoint, stage1_early_stop],
    verbose=1,
)


## 7. Stage 2 — Fine-Tune with Partial Unfreeze
We unfreeze the last 30% of DenseNet121's layers (its deepest, most task-specific feature detectors) while keeping the earlier 70% (generic, low-level ImageNet features) frozen, and drop the learning rate by two orders of magnitude (`1e-3 -> 1e-5`). This is the standard staged transfer-learning recipe: fine-tune only the parts of the backbone likely to hold domain-specific features, at a learning rate small enough not to wreck what the head already learned in Stage 1.


In [ ]:
FINE_TUNE_FRACTION = 0.70  # keep this fraction of layers (from the input side) frozen
fine_tune_at = int(len(base_model.layers) * FINE_TUNE_FRACTION)

base_model.trainable = True
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Unfrozen: {len(base_model.layers) - fine_tune_at} / {len(base_model.layers)} backbone layers")

model.compile(optimizer=Adam(learning_rate=1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

stage2_checkpoint = ModelCheckpoint(
    str(REPO_ROOT / "models" / "densenet_dementia.h5"),
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)
stage2_early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)

(REPO_ROOT / "models").mkdir(parents=True, exist_ok=True)

history_stage2 = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=[stage2_checkpoint, stage2_early_stop],
    verbose=1,
)


## 8. Learning Curves
Accuracy/loss across both stages, with a marker at the Stage 1 -> Stage 2 transition so you can see the effect of unfreezing.


In [ ]:
stage1_epochs = len(history_stage1.history["loss"])

acc = history_stage1.history["accuracy"] + history_stage2.history["accuracy"]
val_acc = history_stage1.history["val_accuracy"] + history_stage2.history["val_accuracy"]
loss = history_stage1.history["loss"] + history_stage2.history["loss"]
val_loss = history_stage1.history["val_loss"] + history_stage2.history["val_loss"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, train_vals, val_vals, title in [
    (axes[0], acc, val_acc, "Accuracy"),
    (axes[1], loss, val_loss, "Loss"),
]:
    ax.plot(train_vals, label="Train")
    ax.plot(val_vals, label="Validation")
    ax.axvline(stage1_epochs - 0.5, color="gray", linestyle="--", label="Stage 1 -> Stage 2")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()
plt.show()


## 9. Evaluate on the Test Split


In [ ]:
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss    : {test_loss:.4f}")

test_generator.reset()
probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(probs, axis=1)
y_true = np.array(test_generator.classes).astype(int)


## 10. Manual Scott's Pi & Quadratic Weighted Kappa
Using the same `src/evaluate.py` functions as the CNN and YOLOv8 benchmarks, so all three models' numbers come from identical metric code.


In [ ]:
densenet_metrics = evaluate_predictions(y_true, y_pred, num_classes=4)

DOCUMENTED_TARGETS = {
    "Custom 6-Block CNN": {"accuracy": 0.7573, "scotts_pi": 0.4257, "qwk": 0.6680},
    "YOLOv8": {"accuracy": 0.9164, "scotts_pi": 0.7869, "qwk": 0.8447},
}

summary = pd.DataFrame([
    {"Model": "Custom 6-Block CNN", **DOCUMENTED_TARGETS["Custom 6-Block CNN"]},
    {"Model": "YOLOv8", **DOCUMENTED_TARGETS["YOLOv8"]},
    {"Model": "DenseNet121 (staged transfer learning)", **densenet_metrics},
])
print(summary.to_string(index=False))

print(
    "\nUpdate the resume bullet's [X]% with the accuracy above: "
    f"{densenet_metrics['accuracy'] * 100:.2f}%"
)


## 11. Confusion Matrix & Classification Report


In [ ]:
cm = confusion_matrix(y_true, y_pred)
short_names = ["Non\nDemented", "Very Mild", "Mild", "Moderate"]

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=short_names, yticklabels=short_names)
plt.title(f"DenseNet121 — Accuracy={densenet_metrics['accuracy']:.4f}  Scott's Pi={densenet_metrics['scotts_pi']:.4f}")
plt.xlabel("Predicted Stage")
plt.ylabel("True Stage")
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
